![Module 4: Self-Learning Memory](../images/module-4-memory.svg)

# Module 4: Self-Learning Memory

> Amazon Bedrock AgentCore Memory extracts facts from conversations and retrieves them across sessions. Scoped by `actorId` — safe for multi-tenant apps.

📖 Full explainer: see the companion page in `content/` of this repo.  
💻 Standalone script: `code/step-0?/agent.py`  
🔗 Workshop: https://github.com/strands-agents/samples/tree/main/python/01-learn/18-self-improving-agents

---


# Module 3: Long-term Memory (AgentCore Memory)

This step adds long-term memory via Amazon Bedrock AgentCore Memory. Run `code/step-03-memory/create_memory.py` once to create a memory store, then set `BEDROCK_AGENTCORE_MEMORY_ID` below.

In [ ]:
%pip install -q strands-agents strands-agents-tools bedrock-agentcore

## Configure memory + model

Set `BEDROCK_AGENTCORE_MEMORY_ID` to the ID printed by `create_memory.py`.

In [ ]:
import os

MODEL_ID  = "global.anthropic.claude-opus-4-8"  # Claude Sonnet 4.5
REGION    = "us-east-1"
# Run code/step-03-memory/create_memory.py first, then paste the ID here
# (or export BEDROCK_AGENTCORE_MEMORY_ID before launching Jupyter).
MEMORY_ID = os.environ.get("BEDROCK_AGENTCORE_MEMORY_ID", "<your-memory-id>")
os.environ["AWS_REGION"] = REGION
os.environ["BEDROCK_AGENTCORE_MEMORY_ID"] = MEMORY_ID
print(f"Memory: {MEMORY_ID}")

## Build the memory-aware agent

In [ ]:
from strands import Agentfrom strands_tools import shellfrom bedrock_agentcore.memory.integrations.strands.config import AgentCoreMemoryConfig, RetrievalConfigfrom bedrock_agentcore.memory.integrations.strands.session_manager import AgentCoreMemorySessionManagerfrom strands_tools.agent_core_memory import AgentCoreMemoryToolProviderACTOR_ID = os.environ.get("USER", "attendee")SESSION_ID = f"aim308-{ACTOR_ID}"cfg = AgentCoreMemoryConfig(    memory_id=MEMORY_ID, session_id=SESSION_ID, actor_id=ACTOR_ID,    retrieval_config={        f"/users/{ACTOR_ID}/facts":       RetrievalConfig(top_k=3, relevance_score=0.5),        f"/users/{ACTOR_ID}/preferences": RetrievalConfig(top_k=3, relevance_score=0.5),    },)memory_tools = AgentCoreMemoryToolProvider(    memory_id=MEMORY_ID, session_id=SESSION_ID,    actor_id=ACTOR_ID, namespace="default", region=REGION,).toolsdef memory_agent():    return Agent(        model=MODEL_ID,        tools=[shell] + memory_tools,        session_manager=AgentCoreMemorySessionManager(cfg, REGION),        system_prompt=(            "You are a research agent with long-term memory. "            "Retrieve before answering. Save facts and preferences."        ),    )

## Teach it about yourself

In [ ]:
memory_agent()("My name is Sam. I prefer Python. I'm building agricultural drones.")

## New agent instance, same actor — it remembers

In [ ]:
memory_agent()("what do you remember about me?")

Next: **[04_autonomous.ipynb](04_autonomous.ipynb)**